# Modelling — Mortgage Default (IFRS-9 PD Estimation)

Pipeline: Dummy → Logistic (with class weights) → LDA → Naive Bayes → Decision Tree → Random Forest → XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)


## Load Data

In [ ]:
import pickle
from joblib import load as joblib_load

# BUG FIX: original Cell 2 loaded test_df.csv into train_df (copy-paste error).
# Corrected: each split loads its own file.
BASE = r"C:\Users\Niraj Mhatre\projects\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\notebooks\\"

train_df = pd.read_csv(BASE + "train_df.csv")
val_df   = pd.read_csv(BASE + "val_df.csv")
test_df  = pd.read_csv(BASE + "test_df.csv")

X_train_pca = pd.read_csv(BASE + "X_train_pca.csv")
X_val_pca   = pd.read_csv(BASE + "X_val_pca.csv")
X_test_pca  = pd.read_csv(BASE + "X_test_pca.csv")

try:
    with open(BASE + "pca_model.pkl", "rb") as f:
        pca_model = pickle.load(f)
except Exception:
    pca_model = joblib_load(BASE + "pca_model.pkl")

# Hard guard: catch train/test file mixup immediately
assert not train_df.equals(test_df), \
    "train_df and test_df are identical — check file paths above."
assert not train_df.equals(val_df), \
    "train_df and val_df are identical — check file paths above."

print("Shapes — train:", train_df.shape, "| val:", val_df.shape, "| test:", test_df.shape)


## Feature / Target Split + Data Quality Guard

In [ ]:
TARGET = "defaulted_flag"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val   = val_df.drop(columns=[TARGET])
y_val   = val_df[TARGET]

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

# Class balance
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name}: default rate = {y.mean():.4f} ({y.sum():,} / {len(y):,})")

# BUG FIX: original code had X_train.dropna() which silently desynced
# X_train and y_train row counts, causing shape-mismatch errors later.
# The feature-engineering notebook guarantees 0 NaN/Inf; assert here instead.
for name, X in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    n_nan = X.isna().sum().sum()
    n_inf = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
    assert n_nan == 0, f"{name}: {n_nan} NaNs — re-run feature engineering"
    assert n_inf == 0, f"{name}: {n_inf} Infs — re-run feature engineering"
print("\nAll splits: 0 NaN, 0 Inf — ready for modelling.")


## Evaluation Utility

In [ ]:
def evaluate(model, X, y, split_name="Val", threshold=0.5, model_name=""):
    """Print classification metrics and plot confusion matrix + ROC curve."""
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{'='*55}")
    print(f"  {model_name} — {split_name} set  (threshold={threshold})")
    print(f"{'='*55}")
    print(f"  Accuracy : {accuracy_score(y, y_pred):.4f}")
    print(f"  Precision: {precision_score(y, y_pred, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y, y_pred, zero_division=0):.4f}")
    print(f"  F1 Score : {f1_score(y, y_pred, zero_division=0):.4f}")
    print(f"  ROC AUC  : {roc_auc_score(y, y_prob):.4f}")
    print()
    print(classification_report(y, y_pred, zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[0])
    axes[0].set_title(f"{model_name} Confusion Matrix ({split_name})")
    RocCurveDisplay.from_predictions(y, y_prob, ax=axes[1])
    axes[1].set_title(f"{model_name} ROC Curve ({split_name})")
    plt.tight_layout()
    plt.show()

    return roc_auc_score(y, y_prob)


## Dummy Classifier (Baseline)

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)
y_prob_dummy = dummy.predict_proba(X_val)[:, 1]

print("Dummy — Val Accuracy:", accuracy_score(y_val, y_pred_dummy))
print("Dummy — Val ROC AUC :", roc_auc_score(y_val, y_prob_dummy))
print("(Dummy always predicts majority class; ROC AUC = 0.5 expected)")


## Logistic Regression

### Assumptions Checklist
1. Binary target ✓
2. Independent observations — temporal split reduces overlap ✓
3. No perfect multicollinearity — verified via VIF in FE notebook ✓
4. Linear relationship between predictors and log-odds — approximately ✓ after scaling
5. Sufficient sample size ✓ (350k train)

> **Class imbalance** (train=3.2% vs val=9%): `class_weight='balanced'` used to prevent the model from predicting all non-defaults.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000,
    # BUG FIX: original code had no class_weight despite 3% default rate.
    # Without this, LR collapses to predicting 0 for everyone (accuracy=97%,
    # recall=0%), which is useless for IFRS-9 PD estimation.
    class_weight="balanced",
    solver="lbfgs",
    C=1.0,
)

lr.fit(X_train, y_train)
print("Logistic Regression trained.")


In [ ]:
# BUG FIX: original code predicted on X_val but scored against y_test
# (size mismatch: 50k vs 150k rows → guaranteed ValueError crash).
# Corrected: evaluate on val set only here; test set evaluated once at the end.
lr_val_auc = evaluate(lr, X_val, y_val, split_name="Val", model_name="Logistic Regression")


In [ ]:
# Coefficient inspection
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr.coef_[0],
    "Odds Ratio": np.exp(lr.coef_[0])
}).sort_values("Coefficient", key=abs, ascending=False)
print(coef_df.to_string(index=False))


## Linear Discriminant Analysis

LDA assumes multivariate normality within classes and equal covariance matrices. Neither holds perfectly here (skewed WOE and interaction terms), but LDA is robust to mild violations and provides a useful discriminant benchmark. `class_weight` not directly available in LDA — we use `priors` to correct for the class imbalance.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Use prior = empirical class frequencies from train
pos_rate = y_train.mean()
lda = LinearDiscriminantAnalysis(priors=[1 - pos_rate, pos_rate])
lda.fit(X_train, y_train)

lda_val_auc = evaluate(lda, X_val, y_val, split_name="Val", model_name="LDA")


## Gaussian Naive Bayes

Assumes features are conditionally independent given class — a strong assumption that clearly doesn't hold (WOE columns correlate with raw features), but NB often has decent ROC AUC even with violated independence. Use `var_smoothing` to stabilise near-zero-variance features.

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB(var_smoothing=1e-9)
nb.fit(X_train, y_train)

nb_val_auc = evaluate(nb, X_val, y_val, split_name="Val", model_name="Naive Bayes")


## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=200,
    class_weight="balanced",
    random_state=42,
)
dt.fit(X_train, y_train)

dt_val_auc = evaluate(dt, X_val, y_val, split_name="Val", model_name="Decision Tree")


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=100,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_val_auc = evaluate(rf, X_val, y_val, split_name="Val", model_name="Random Forest")


In [ ]:
# Feature importance
fi = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)
print(fi.to_string(index=False))


## XGBoost

`scale_pos_weight = n_negatives / n_positives` corrects class imbalance in the gradient boosting objective, equivalent to oversampling the minority class.

In [ ]:
try:
    from xgboost import XGBClassifier
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    spw   = n_neg / n_pos   # scale_pos_weight corrects class imbalance

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=spw,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_val_auc = evaluate(xgb, X_val, y_val, split_name="Val", model_name="XGBoost")
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    xgb_val_auc = None


## Model Comparison — Validation Set ROC AUC

In [ ]:
results = {
    "Dummy":           0.5,
    "Logistic Reg":    lr_val_auc,
    "LDA":             lda_val_auc,
    "Naive Bayes":     nb_val_auc,
    "Decision Tree":   dt_val_auc,
    "Random Forest":   rf_val_auc,
}
if xgb_val_auc is not None:
    results["XGBoost"] = xgb_val_auc

summary = pd.DataFrame(results.items(), columns=["Model", "Val ROC AUC"]).sort_values("Val ROC AUC", ascending=False)
print(summary.to_string(index=False))

plt.figure(figsize=(8,4))
plt.barh(summary["Model"], summary["Val ROC AUC"])
plt.axvline(0.5, color='red', linestyle='--', label='Random baseline')
plt.xlabel("ROC AUC"); plt.title("Model Comparison — Validation ROC AUC")
plt.legend(); plt.tight_layout(); plt.show()


## Final Test-Set Evaluation

> **Run this cell only once** after model selection is finalised on val. Re-tuning based on test results constitutes test-set leakage.

Replace `best_model` with whichever model won on val.

In [ ]:
# Replace with whichever model had the best Val ROC AUC above
best_model      = rf        # ← change this after reviewing val results
best_model_name = "Random Forest"

test_auc = evaluate(
    best_model, X_test, y_test,
    split_name="Test (held-out)",
    model_name=best_model_name
)
print(f"\nFinal held-out Test ROC AUC: {test_auc:.4f}")
